In [1]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from simulator import smFRET_simulator_wrapper
from gsl_spline import spline
from priors import generate_spline_fret_prior_from_config
from utils import make_spline_knots

/home/dingeldein/Venvs/fret_sbi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'generate_spline_fret_prior_from_config' from 'priors' (/home/dingeldein/Desktop/fret_sbi/smFRET_sbi/src/priors.py)

In [7]:
import time
import torch
from torch.utils.data import DataLoader
from data_loaders import H5Dataset

def benchmark(chunk_size, chunk_step, num_workers, prefetch_factor, num_batches=100):
    dataset = H5Dataset(
        "../data/base/data_compressed.h5",
        batch_size=128,
        chunk_size=chunk_size,
        chunk_step=chunk_step,
        shuffle=True
    )
    
    loader = DataLoader(
        dataset,
        batch_size=None,
        num_workers=num_workers,
        prefetch_factor=prefetch_factor,
        pin_memory=True
    )
    
    start = time.perf_counter()
    for i, (theta, x) in enumerate(loader):
        if i == 0:
            print(f"Batch {i+1}: theta shape={theta.shape}, x shape={x.shape}")
        x = x.cuda(non_blocking=True)  # Simulate GPU transfer
        if i >= num_batches:
            break
    elapsed = time.perf_counter() - start
    
    print(f"chunk_size={chunk_size}, chunk_step={chunk_step}, workers={num_workers}, prefetch={prefetch_factor}")
    print(f"  → {num_batches / elapsed:.1f} batches/sec")

In [11]:
# Test combinations
benchmark(chunk_size=1000, chunk_step=1,  num_workers=8, prefetch_factor=2)

Batch 1: theta shape=torch.Size([128, 6]), x shape=torch.Size([128, 80000, 2])
chunk_size=1000, chunk_step=1, workers=8, prefetch=2
  → 15.6 batches/sec


In [21]:
benchmark(chunk_size=1000, chunk_step=1,  num_workers=8, prefetch_factor=2)

Batch 1: theta shape=torch.Size([128, 6]), x shape=torch.Size([128, 80000, 2])
chunk_size=1000, chunk_step=1, workers=8, prefetch=2
  → 15.8 batches/sec


In [10]:
benchmark(chunk_size=128, chunk_step=1, num_workers=6, prefetch_factor=3)

Batch 1: theta shape=torch.Size([128, 6]), x shape=torch.Size([128, 80000, 2])
chunk_size=128, chunk_step=1, workers=6, prefetch=3
  → 1.4 batches/sec


In [ ]:
# Test combinations
benchmark(chunk_step=64,  num_workers=6, prefetch_factor=1)
benchmark(chunk_step=128, num_workers=6, prefetch_factor=1)
benchmark(chunk_step=128, num_workers=4, prefetch_factor=2)
benchmark(chunk_step=128, num_workers=6, prefetch_factor=2)

In [2]:
@np.vectorize
def double_well_potential(x, deltaG=4.0, deltax=2.4):
    x = x - 5.0
    x_rel = x / deltax
    if np.abs(x_rel) >= 0 and np.abs(x_rel) <= 0.5:
        g =  -2*x_rel**2
    elif np.abs(x_rel) > 0.5:
        g =  2*(np.abs(x_rel) - 1)**2 - 1
    return g * deltaG

In [4]:
x_min, x_max = 4.0, 14
num_knots = 5
x = np.linspace(x_min, x_max, num_knots + 4) # +4 for boundary nodes
y = double_well_potential(x, deltaG=5) # +4 for boundary nodes
x = x[2:-2]
y = y[2:-2]

In [5]:
def eval_spline(params, N_knots, x_min, x_max, boundary_condition, max_G0, spline_type="cubic"):
    
    
    x_knots, y_knots = make_spline_knots(
        params=params,
        x_min=x_min,
        x_max=x_max,
        num_knots=N_knots,
        max_G0=max_G0,
        boundary_condition=boundary_condition
    )

    _x =  np.linspace(x_min, x_max, 1000)
    _y = spline(x_knots, y_knots, _x, spline_type=spline_type)

    return _x, _y

In [6]:
params = torch.tensor([0.0, *y, 4.0, 10.5, 0.05, 0.05, 0.75, 0.75, 0.9, 0.9, 0.1, 0.1])

In [7]:
params

tensor([ 0.0000, -3.5938, -4.7873, -0.5556,  9.1016, 24.1840,  4.0000, 10.5000,
         0.0500,  0.0500,  0.7500,  0.7500,  0.9000,  0.9000,  0.1000,  0.1000],
       dtype=torch.float64)

In [ ]:
plt.plot(*eval_spline(params, N_knots=num_knots, x_min=x_min, x_max=x_max, boundary_condition='bound', max_G0=50.0, spline_type="cubic"), color='blue')
plt.scatter(x, y, color='red')
plt.ylim(-10, 1)

In [ ]:
observation = smFRET_simulator_wrapper(                
    params, 
    dt=5e-4, 
    saving_freq=100, 
    T=5000.0, 
    x_init=(3.0, 7.0),
    x_min=x_min,
    x_max=x_max,
    N_knots=num_knots,
    max_G0=30.0,
    boundary_condition='bound',
    spline_type="steffen"
)

In [ ]:
type(observation)

In [ ]:
def bin_along_time(x, n_bins=100):
    n = len(x)
    n_per_bin = int(n / n_bins)
    x_binned = np.zeros(n_bins)
    for i in range(n_bins):
        x_binned[i] = np.sum(x[i*n_per_bin:(i+1)*n_per_bin])
    return x_binned

In [ ]:
plt.figure(figsize=(7, 2.5))
plt.plot(bin_along_time(observation[ :,0].numpy(), n_bins=155), label="Donor")
plt.plot(bin_along_time(observation[ :,1].numpy(), n_bins=155), label="acceptor")
plt.ylabel("Photon count")
plt.xlabel("Time")


In [ ]:
from priors import generate_spline_fret_prior_from_config
from utils import eval_spline

In [ ]:
prior = generate_spline_fret_prior_from_config("../conf/simulator/default.yaml")

In [ ]:
params = prior.sample((40,))

In [ ]:
for i in range(6):
    x, y = eval_spline(
        params[i],
        x_eval=np.linspace(0, 15, 100), 
        N_knots=5,
        x_min=4.0, 
        x_max=14.0, 
        boundary_condition='bound', 
        max_G0=30.0, 
        spline_type="cubic"
    )
    plt.plot(x, y, color='blue', alpha=0.4)
plt.ylim(-10, 10)

In [ ]:
x, y = eval_spline(
    params[23],
    x_eval=np.linspace(0, 15, 100), 
    N_knots=5,
    x_min=4.0, 
    x_max=14.0, 
    boundary_condition='bound', 
    max_G0=30.0, 
    spline_type="cubic"
)
plt.plot(x, y, color='blue', alpha=0.4)

In [ ]:
params[23][2+num_knots] = 0.1
observation = smFRET_simulator_wrapper(                
    params[23], 
    dt=5e-5, 
    saving_freq=1000, 
    T=10000.0, 
    x_init=(5.0, 15.0),
    x_min=x_min,
    x_max=x_max,
    N_knots=num_knots,
    max_G0=30.0,
    boundary_condition='bound',
    spline_type="cubic"
)

In [ ]:
plt.figure(figsize=(7, 2.5))
plt.plot(bin_along_time(observation[ :,0].numpy(), n_bins=155), label="Donor")
plt.plot(bin_along_time(observation[ :,1].numpy(), n_bins=155), label="acceptor")
plt.ylabel("Photon count")
plt.xlabel("Time")